
# Three-Stage Framework vs OAENet — Python Version (No `statsmodels`)

This version removes the `statsmodels` dependency completely.

It only requires:

```bash
pip install numpy pandas scipy scikit-learn
```

The ATT regression after matching is implemented with `sklearn.linear_model.LinearRegression`, so the earlier SciPy/Statsmodels compatibility error will not occur.


In [ ]:

# If needed, run this once:
# %pip install -U numpy pandas scipy scikit-learn

import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC

print("Imports successful.")


In [ ]:

CONFIG = {
    "p": 20,
    "true_te": 0.5,
    "alpha_enet": 0.5,
    "nfolds": 5,
    "gamma1_elrt": 0.5,
    "gamma1_esvms": 1.0,
    "gamma2": 1.0,
    "coef_tol": 1e-7,
    "treatment_mode": "bernoulli_logit",
}


## 1. Synthetic data-generating process

In [ ]:

def compound_sigma(p, rho):
    S = np.full((p, p), rho, dtype=float)
    np.fill_diagonal(S, 1.0)
    return S


def scenario_coefficients(scenario, p=20):
    theta = np.zeros(p)
    beta = np.zeros(p)

    if scenario == 1:
        theta[:4] = [.6, .6, .6, .6]
        beta[[0, 1, 4, 5]] = [1, 1, 1, 1]
    elif scenario == 2:
        theta[:4] = [.6, .6, .6, .6]
        beta[[0, 1, 4, 5]] = [.4, .4, 1, 1]
    elif scenario == 3:
        theta[:4] = [.2, .2, .6, .6]
        beta[[0, 1, 4, 5]] = [1, 1, 1, 1]
    elif scenario == 4:
        theta[:4] = [.6, .6, .6, .6]
        beta[[0, 1, 4, 5]] = [1, 1, 1.8, 1.8]
    else:
        raise ValueError("scenario must be 1, 2, 3, or 4")

    return theta, beta


def simulate_scenario(n, rho, scenario, seed, treatment_mode=None):
    if treatment_mode is None:
        treatment_mode = CONFIG["treatment_mode"]

    rng = np.random.default_rng(seed)
    p = CONFIG["p"]

    theta, beta = scenario_coefficients(scenario, p)

    X = rng.multivariate_normal(
        mean=np.zeros(p),
        cov=compound_sigma(p, rho),
        size=n
    )

    eta = X @ beta

    if treatment_mode == "bernoulli_logit":
        T = rng.binomial(1, expit(eta))
    elif treatment_mode == "paper_literal":
        eps_treat = rng.normal(size=n)
        T = (expit(eta) < eps_treat).astype(int)
    else:
        raise ValueError("unknown treatment_mode")

    Y = CONFIG["true_te"] * T + X @ theta + rng.normal(size=n)

    df = pd.DataFrame(
        X,
        columns=[f"X{i}" for i in range(1, p + 1)]
    )
    df.insert(0, "T", T)
    df.insert(0, "Y", Y)

    return df


## 2. Weighted elastic-net solvers

In [ ]:

def soft_threshold(x, threshold):
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0.0)


def gaussian_weighted_enet(
    X, y, w, lam, alpha,
    init=None, max_iter=2000, tol=1e-6
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    n, p = X.shape

    if init is None:
        beta0 = y.mean()
        beta = np.zeros(p)
    else:
        beta0, beta = init
        beta = beta.copy()

    spectral = np.linalg.norm(X, 2)
    L = (spectral ** 2) / n + lam * (1 - alpha)
    L = max(L, 1e-8)

    z0 = beta0
    z = beta.copy()
    t = 1.0

    for _ in range(max_iter):
        residual = (z0 + X @ z) - y

        grad0 = residual.mean()
        grad = X.T @ residual / n + lam * (1 - alpha) * z

        beta0_new = z0 - grad0 / L
        beta_new = soft_threshold(
            z - grad / L,
            lam * alpha * w / L
        )

        delta = max(
            abs(beta0_new - beta0),
            np.max(np.abs(beta_new - beta))
        )

        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        momentum = (t - 1) / t_new

        z0 = beta0_new + momentum * (beta0_new - beta0)
        z = beta_new + momentum * (beta_new - beta)

        beta0 = beta0_new
        beta = beta_new
        t = t_new

        if delta < tol:
            break

    return beta0, beta


def logistic_weighted_enet(
    X, y, w, lam, alpha,
    init=None, max_iter=2000, tol=1e-6
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    n, p = X.shape

    if init is None:
        mu = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        beta0 = np.log(mu / (1 - mu))
        beta = np.zeros(p)
    else:
        beta0, beta = init
        beta = beta.copy()

    spectral = np.linalg.norm(X, 2)
    L = (spectral ** 2) / (4 * n) + lam * (1 - alpha)
    L = max(L, 0.25)

    z0 = beta0
    z = beta.copy()
    t = 1.0

    for _ in range(max_iter):
        p_hat = expit(z0 + X @ z)
        residual = p_hat - y

        grad0 = residual.mean()
        grad = X.T @ residual / n + lam * (1 - alpha) * z

        beta0_new = z0 - grad0 / L
        beta_new = soft_threshold(
            z - grad / L,
            lam * alpha * w / L
        )

        delta = max(
            abs(beta0_new - beta0),
            np.max(np.abs(beta_new - beta))
        )

        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        momentum = (t - 1) / t_new

        z0 = beta0_new + momentum * (beta0_new - beta0)
        z = beta_new + momentum * (beta_new - beta)

        beta0 = beta0_new
        beta = beta_new
        t = t_new

        if delta < tol:
            break

    return beta0, beta


def cv_weighted_enet(
    X, y, w,
    family="gaussian",
    alpha=0.5,
    nfolds=5,
    seed=1,
    nlambda=25
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    lambdas = np.geomspace(1.0, 1e-4, nlambda)

    if family == "binomial":
        splitter = StratifiedKFold(
            n_splits=nfolds,
            shuffle=True,
            random_state=seed
        )
        split_iter = splitter.split(X, y.astype(int))
    else:
        splitter = KFold(
            n_splits=nfolds,
            shuffle=True,
            random_state=seed
        )
        split_iter = splitter.split(X)

    losses = np.zeros((nfolds, nlambda))

    for fold, (tr, va) in enumerate(split_iter):
        init = None

        for j, lam in enumerate(lambdas):
            if family == "gaussian":
                b0, b = gaussian_weighted_enet(
                    X[tr], y[tr], w, lam, alpha, init=init
                )
                pred = b0 + X[va] @ b
                losses[fold, j] = np.mean((y[va] - pred) ** 2)

            elif family == "binomial":
                b0, b = logistic_weighted_enet(
                    X[tr], y[tr], w, lam, alpha, init=init
                )
                p = np.clip(
                    expit(b0 + X[va] @ b),
                    1e-10,
                    1 - 1e-10
                )
                losses[fold, j] = -2 * np.mean(
                    y[va] * np.log(p)
                    + (1 - y[va]) * np.log(1 - p)
                )
            else:
                raise ValueError("family must be gaussian or binomial")

            init = (b0, b)

    mean_loss = losses.mean(axis=0)
    se_loss = losses.std(axis=0, ddof=1) / np.sqrt(nfolds)

    j_min = np.argmin(mean_loss)
    cutoff = mean_loss[j_min] + se_loss[j_min]

    eligible = np.where(mean_loss <= cutoff)[0]
    j_1se = eligible[0]

    lam_1se = lambdas[j_1se]

    if family == "gaussian":
        b0, b = gaussian_weighted_enet(
            X, y, w, lam_1se, alpha
        )
    else:
        b0, b = logistic_weighted_enet(
            X, y, w, lam_1se, alpha
        )

    return {
        "intercept": b0,
        "coef": b,
        "lambda_1se": lam_1se
    }


## 3. Proposed three-stage framework

In [ ]:

def safe_inverse_power(x, gamma, eps=1e-8, cap=1e12):
    return np.minimum(
        np.maximum(np.abs(x), eps) ** (-gamma),
        cap
    )


def proposed_three_stage(
    df,
    prior="LR",
    smoother="tanh",
    gamma1=None,
    gamma2=1.0,
    seed=1
):
    variables = [f"X{i}" for i in range(1, CONFIG["p"] + 1)]

    X = df[variables].to_numpy(dtype=float)
    Y = df["Y"].to_numpy(dtype=float)
    T = df["T"].to_numpy(dtype=int)

    Xs = StandardScaler().fit_transform(X)

    # Stage 1: exposure model
    if prior == "LR":
        # Very large C approximates an unpenalized logistic regression
        model = LogisticRegression(
            penalty="l2",
            C=1e6,
            solver="lbfgs",
            max_iter=5000
        )
        model.fit(Xs, T)
        beta = model.coef_.ravel()

    elif prior == "SVM":
        # n > p in the published experiments, so dual=False is appropriate
        model = LinearSVC(
            C=1.0,
            dual=False,
            max_iter=10000
        )
        model.fit(Xs, T)
        beta = model.coef_.ravel()

    else:
        raise ValueError("prior must be LR or SVM")

    if gamma1 is None:
        gamma1 = (
            CONFIG["gamma1_elrt"]
            if smoother == "tanh"
            else CONFIG["gamma1_esvms"]
        )

    if smoother == "tanh":
        w1 = np.maximum(
            np.tanh(np.abs(beta)),
            1e-8
        ) ** gamma1

    elif smoother == "sigmoid":
        w1 = (
            1 / (1 + np.exp(-np.abs(beta)))
        ) ** gamma1

    else:
        raise ValueError("smoother must be tanh or sigmoid")

    # Stage 2
    fit2 = cv_weighted_enet(
        Xs,
        Y,
        w1,
        family="gaussian",
        alpha=CONFIG["alpha_enet"],
        nfolds=CONFIG["nfolds"],
        seed=seed
    )
    theta = fit2["coef"]

    # Stage 3 adaptive weights
    psi = safe_inverse_power(
        theta,
        gamma2
    )

    fit3 = cv_weighted_enet(
        Xs,
        Y,
        psi,
        family="gaussian",
        alpha=CONFIG["alpha_enet"],
        nfolds=CONFIG["nfolds"],
        seed=seed + 10000
    )

    final_beta = fit3["coef"]

    selected = [
        variables[i]
        for i in np.flatnonzero(
            np.abs(final_beta) > CONFIG["coef_tol"]
        )
    ]

    return {
        "selected": selected,
        "beta_prior": beta,
        "penalty_stage2": w1,
        "theta_stage2": theta,
        "penalty_stage3": psi,
        "theta_final": final_beta
    }


def fit_enh_elrt(df, seed=1):
    return proposed_three_stage(
        df,
        prior="LR",
        smoother="tanh",
        gamma1=CONFIG["gamma1_elrt"],
        gamma2=CONFIG["gamma2"],
        seed=seed
    )


def fit_enh_esvms(df, seed=1):
    return proposed_three_stage(
        df,
        prior="SVM",
        smoother="sigmoid",
        gamma1=CONFIG["gamma1_esvms"],
        gamma2=CONFIG["gamma2"],
        seed=seed
    )


## 4. OAENet comparator

In [ ]:

def fit_oaenet(df, gamma=3, seed=1):
    variables = [f"X{i}" for i in range(1, CONFIG["p"] + 1)]

    X = df[variables].to_numpy(dtype=float)
    Y = df["Y"].to_numpy(dtype=float)
    T = df["T"].to_numpy(dtype=int)

    Xs = StandardScaler().fit_transform(X)

    # Stage 1: OLS outcome model
    theta = LinearRegression().fit(Xs, Y).coef_

    # Adaptive weights
    w = safe_inverse_power(
        theta,
        gamma
    )

    # Stage 2: adaptive elastic-net treatment model
    fit = cv_weighted_enet(
        Xs,
        T,
        w,
        family="binomial",
        alpha=CONFIG["alpha_enet"],
        nfolds=CONFIG["nfolds"],
        seed=seed
    )

    beta_t = fit["coef"]

    selected = [
        variables[i]
        for i in np.flatnonzero(
            np.abs(beta_t) > CONFIG["coef_tol"]
        )
    ]

    return {
        "selected": selected,
        "theta_ols": theta,
        "penalty": w,
        "beta_treatment": beta_t
    }


## 5. 1:1 nearest-neighbor matching + ATT regression

In [ ]:

def estimate_att(df, selected):
    if len(selected) == 0:
        return np.nan

    X = df[selected].to_numpy(dtype=float)
    T = df["T"].to_numpy(dtype=int)

    # Approximate unpenalized logistic PS model
    ps_model = LogisticRegression(
        penalty="l2",
        C=1e6,
        solver="lbfgs",
        max_iter=5000
    )
    ps_model.fit(X, T)

    ps = ps_model.predict_proba(X)[:, 1]

    eps = 1e-8
    ps = np.clip(ps, eps, 1 - eps)
    logit_ps = np.log(ps / (1 - ps))

    treated = np.where(T == 1)[0]
    controls = set(np.where(T == 0)[0].tolist())

    pairs = []

    # Greedy 1:1 matching without replacement
    for ti in treated[np.argsort(-logit_ps[treated])]:

        if len(controls) == 0:
            break

        control_idx = np.array(
            sorted(controls),
            dtype=int
        )

        nearest_pos = np.argmin(
            np.abs(
                logit_ps[control_idx]
                - logit_ps[ti]
            )
        )

        ci = int(control_idx[nearest_pos])

        pairs.append((ti, ci))
        controls.remove(ci)

    if len(pairs) == 0:
        return np.nan

    matched_rows = [
        idx
        for pair in pairs
        for idx in pair
    ]

    matched = df.iloc[
        matched_rows
    ].copy()

    # Post-matching regression:
    # Y ~ T + selected covariates
    Xreg = matched[
        ["T"] + selected
    ].to_numpy(dtype=float)

    yreg = matched[
        "Y"
    ].to_numpy(dtype=float)

    reg = LinearRegression()
    reg.fit(Xreg, yreg)

    # First coefficient corresponds to T
    return float(reg.coef_[0])


## 6. One-run sanity check

In [ ]:

df_test = simulate_scenario(
    n=500,
    rho=0.25,
    scenario=1,
    seed=1
)

fit1 = fit_enh_elrt(df_test, seed=1)
fit2 = fit_enh_esvms(df_test, seed=1)
fit3 = fit_oaenet(df_test, seed=1)

print("Enh-ELRT:", fit1["selected"])
print("Enh-ESVMS:", fit2["selected"])
print("OAENet:", fit3["selected"])

print()
print("ATT Enh-ELRT:", estimate_att(df_test, fit1["selected"]))
print("ATT Enh-ESVMS:", estimate_att(df_test, fit2["selected"]))
print("ATT OAENet:", estimate_att(df_test, fit3["selected"]))
print("ATT Target:", estimate_att(df_test, ["X1","X2","X3","X4"]))


## 7. Direct comparison function

In [ ]:

def run_one_comparison(
    n=500,
    rho=0.25,
    scenario=1,
    seed=1,
    treatment_mode=None
):
    df = simulate_scenario(
        n,
        rho,
        scenario,
        seed,
        treatment_mode
    )

    fits = {
        "Enh_ELRT": fit_enh_elrt(df, seed),
        "Enh_ESVMS": fit_enh_esvms(df, seed),
        "OAENet": fit_oaenet(df, seed=seed)
    }

    target = ["X1", "X2", "X3", "X4"]
    target_att = estimate_att(
        df,
        target
    )

    rows = []

    for method, fit in fits.items():

        selected = fit["selected"]
        att = estimate_att(
            df,
            selected
        )

        rows.append({
            "method": method,
            "n_selected": len(selected),
            "selected": ",".join(selected),
            "att": att,
            "target_att": target_att,
            "abs_selection_bias": abs(att - target_att),
            "pct_selection_bias":
                100 * abs(att - target_att)
                / CONFIG["true_te"]
        })

    rows.append({
        "method": "Target",
        "n_selected": 4,
        "selected": ",".join(target),
        "att": target_att,
        "target_att": target_att,
        "abs_selection_bias": 0.0,
        "pct_selection_bias": 0.0
    })

    return pd.DataFrame(rows)


run_one_comparison(
    n=500,
    rho=0.25,
    scenario=1,
    seed=1
)


## 8. Selection frequency

In [ ]:

def selection_frequency(
    n=500,
    rho=0.25,
    scenario=1,
    seeds=range(1,31)
):
    seeds = list(seeds)

    variables = [
        f"X{i}"
        for i in range(
            1,
            CONFIG["p"] + 1
        )
    ]

    methods = [
        "Enh_ELRT",
        "Enh_ESVMS",
        "OAENet"
    ]

    counts = {
        m: pd.Series(
            0.0,
            index=variables
        )
        for m in methods
    }

    for seed in seeds:

        df = simulate_scenario(
            n=n,
            rho=rho,
            scenario=scenario,
            seed=seed
        )

        fits = {
            "Enh_ELRT":
                fit_enh_elrt(df, seed),

            "Enh_ESVMS":
                fit_enh_esvms(df, seed),

            "OAENet":
                fit_oaenet(df, seed=seed)
        }

        for method in methods:
            counts[method].loc[
                fits[method]["selected"]
            ] += 1

    return pd.DataFrame({
        method:
            counts[method] / len(seeds)

        for method in methods
    })


# Example:
# sf = selection_frequency()
# sf
